# Lab 03 challenge: worked solution

For the instructor. It copies the pieces it needs from the lab (the copilot's prompt, `get_policy`, a SQLite
checkpointer and `rows_holding`), then answers the two tasks. Model replies change from run to run; the notes give
this run's values (`gpt-4o-mini`, 13 Oct 2026).

In [1]:
# Setup: on Colab, copy the repository and install the packages. Locally (after uv sync) nothing is installed.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/agentic_ai_conversational_agents").exists():
        !git clone -q --depth 1 https://github.com/tatwan/agentic_ai_conversational_agents.git /content/agentic_ai_conversational_agents
    %cd /content/agentic_ai_conversational_agents
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: agentic_ai_conversational_agents


In [2]:
# Config: model names come from models.env; the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name in ("CHAT_MODEL", "REASONING_EFFORT", "JUDGE_MODEL", "EMBED_MODEL"):
    print(f"{name:18}{settings[name] or '(not set)'}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

CHAT_MODEL        gpt-4o-mini
REASONING_EFFORT  (not set)
JUDGE_MODEL       gpt-4o-mini
EMBED_MODEL       text-embedding-3-small
OPENAI_API_KEY    found in .env


## Setup: the pieces from the lab

The pieces from the lab and the challenge's starting cell. The solution keeps its threads in its own file, `solved/threads_solution.sqlite`, deleted
at the start.

In [3]:
import sqlite3
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver
from shared.settings import chat_model

SYSTEM_PROMPT = (ROOT / "shared" / "prompts" / "copilot_system_v1.md").read_text(encoding="utf-8")
SYSTEM_PROMPT = SYSTEM_PROMPT.split("-->", 1)[1].strip()
SHORT_PROMPT = "You help Chinook support reps. Keep answers short."

CORPUS = ROOT / "data" / "corpus"
POLICIES = sorted(path.stem.removeprefix("policy_") for path in CORPUS.glob("policy_*.md"))

def get_policy(name: str) -> str:
    """Return the full text of one Chinook support policy.

    Call it before answering any question about shop rules.

    Args:
        name: Policy name, e.g. "refunds" or "escalations".
    """
    path = CORPUS / f"policy_{name}.md"
    if not path.exists():
        return f"No policy named {name!r}. Known: {', '.join(POLICIES)}."
    return path.read_text(encoding="utf-8")

policy_tool = tool(get_policy, parse_docstring=True)

THREADS = ROOT / "03_Thread_Memory_Three_Settings" / "solved" / "threads_solution.sqlite"
for old_file in THREADS.parent.glob("threads_solution.sqlite*"):
    old_file.unlink()
conn = sqlite3.connect(THREADS, check_same_thread=False)
checkpointer = SqliteSaver(conn)

def rows_holding(text, thread_id):
    """Print how many rows of each table hold the text, for one thread."""
    for table, column in (("checkpoints", "checkpoint"), ("writes", "value")):
        total, found = conn.execute(
            f"SELECT count(*), sum(instr({column}, ?) > 0) FROM {table} WHERE thread_id = ?",
            (text.encode(), thread_id),
        ).fetchone()
        print(f"{table:11} {found or 0} of {total} rows hold {text!r}")


def show_thread(each_agent, thread_id):
    """Print every message the thread saved, oldest first."""
    for message in each_agent.get_state({"configurable": {"thread_id": thread_id}}).values.get("messages", []):
        print(f"   {message.type:5} | {message.text[:100]}")

## 1. Keep a card number out of the thread

The same copilot as Part E, with `credit_card` and `mask`. `4111 1111 1111 1111` is a standard test number: it passes
the Luhn check that `PIIMiddleware` uses to tell a card number from any 16 digits.

In [4]:
from langchain.agents.middleware import PIIMiddleware

card_copilot = create_agent(chat_model(settings), tools=[policy_tool], system_prompt=SYSTEM_PROMPT,
                       middleware=[PIIMiddleware("credit_card", strategy="mask")], checkpointer=checkpointer)
CARD = "4111 1111 1111 1111"
message = f"She paid with card {CARD} and wants a refund."

for thread_id, options in (("card-default", {}), ("card-exit", {"durability": "exit"})):
    config = {"configurable": {"thread_id": thread_id}}
    card_copilot.invoke({"messages": [("user", message)]}, config, **options)
    print(thread_id, "| saved:", card_copilot.get_state(config).values["messages"][0].text)
    rows_holding(CARD, thread_id)

card-default | saved: She paid with card **** **** **** 1111 and wants a refund.
checkpoints 2 of 9 rows hold '4111 1111 1111 1111'
writes      1 of 13 rows hold '4111 1111 1111 1111'


card-exit | saved: She paid with card **** **** **** 1111 and wants a refund.
checkpoints 0 of 1 rows hold '4111 1111 1111 1111'
writes      0 of 0 rows hold '4111 1111 1111 1111'


The saved message holds `**** **** **** 1111` in both threads. With the default saving the full number is still in 2 checkpoint rows and 1 write row, the input saved before the middleware ran, as with the email address; with `durability="exit"` no row holds it.

The limit: this shows only that this run left no full copy in this new thread's rows. Rows saved earlier keep
theirs, and the mask finds only card numbers that pass the Luhn check.

## 2. One thread per customer

Two short cases, first on one all-day thread, then on one thread each. The agent is Part B's: no tools, the short
prompt, on the SQLite checkpointer.

In [5]:
day_agent = create_agent(chat_model(settings), tools=[], system_prompt=SHORT_PROMPT, checkpointer=checkpointer)
HELENA = ["I'm handling Helena Holý: 7 invoices totalling $49.62.", "What is this customer's invoice total?"]
PUJA = ["Next customer: Puja Srivastava, 6 invoices totalling $36.64.", "What is this customer's invoice total?",
        "Who was my first customer today?"]

def ask(thread_id, turns):
    config = {"configurable": {"thread_id": thread_id}}
    for text in turns:
        result = day_agent.invoke({"messages": [("user", text)]}, config)
        reply = result["messages"][-1]
        print(f"{thread_id:14} | {text[:40]:40} | {reply.usage_metadata['input_tokens']:3} tokens | {reply.text}")

ask("all-day", HELENA + PUJA)
print()
ask("helena-only", HELENA)
ask("puja-only", PUJA)

for thread_id in ("all-day", "puja-only"):
    print(f"\n{thread_id} saved:")
    show_thread(day_agent, thread_id)

all-day        | I'm handling Helena Holý: 7 invoices tot |  38 tokens | Please confirm if Helena has any specific questions or issues regarding the invoices, or if you need assistance with a particular aspect of her account.


all-day        | What is this customer's invoice total?   |  80 tokens | The total for Helena Holý's invoices is $49.62.


all-day        | Next customer: Puja Srivastava, 6 invoic | 122 tokens | The total for Puja Srivastava's invoices is $36.64.


all-day        | What is this customer's invoice total?   | 154 tokens | The total for Puja Srivastava's invoices is $36.64.


all-day        | Who was my first customer today?         | 186 tokens | Your first customer today was Helena Holý.



helena-only    | I'm handling Helena Holý: 7 invoices tot |  38 tokens | Got it! Let me know if you need assistance with anything specific regarding Helena Holý's invoices.


helena-only    | What is this customer's invoice total?   |  73 tokens | The total for Helena Holý's invoices is $49.62.


puja-only      | Next customer: Puja Srivastava, 6 invoic |  42 tokens | Hi Puja, your total for the 6 invoices is $36.64. How can I assist you further?


puja-only      | What is this customer's invoice total?   |  81 tokens | The customer's invoice total is $36.64.


puja-only      | Who was my first customer today?         | 106 tokens | I'm sorry, but I don't have that information. Can I assist you with something else?

all-day saved:
   human | I'm handling Helena Holý: 7 invoices totalling $49.62.
   ai    | Please confirm if Helena has any specific questions or issues regarding the invoices, or if you need
   human | What is this customer's invoice total?
   ai    | The total for Helena Holý's invoices is $49.62.
   human | Next customer: Puja Srivastava, 6 invoices totalling $36.64.
   ai    | The total for Puja Srivastava's invoices is $36.64.
   human | What is this customer's invoice total?
   ai    | The total for Puja Srivastava's invoices is $36.64.
   human | Who was my first customer today?
   ai    | Your first customer today was Helena Holý.

puja-only saved:
   human | Next customer: Puja Srivastava, 6 invoices totalling $36.64.
   ai    | Hi Puja, your total for the 6 invoices is $36.64. How can I assist you further?
   human | Wh

Both set-ups gave each customer's own total, $49.62 and $36.64. On the all-day thread the agent named Helena Holý as
the first customer; on Puja's own thread it could not (*I'm sorry, but I don't have that information.*), which is what
you want: one customer's case never reaches another's. The saved threads show why. The all-day thread holds ten
messages, both cases; Puja's holds six, only hers. So the all-day thread's last call carried both cases, 186 input
tokens against 106, and after 25 customers it would carry 24 other cases. Recommend a thread per customer for an
all-day rep.